In [ ]:
"""
Enhanced SerpAPI LinkedIn Profile Finder
=========================================
Find LinkedIn profile URLs by full name and company name.
Optimized for Google Colab usage.

Setup:
    pip install requests python-dotenv
    Set SERPAPI_API_KEY in environment or pass directly to the class.
"""

import os
import re
import time
import json
import requests
from urllib.parse import quote_plus
from typing import Optional


# ─────────────────────────────────────────────
#  Helper utilities
# ─────────────────────────────────────────────

def _clean(text: str) -> str:
    """Strip extra whitespace."""
    return " ".join(text.split())


def _score_result(result: dict, full_name: str, company: str) -> float:
    """
    Score a single search result 0–100 based on how well it matches
    the target name + company.
    """
    name_lower    = full_name.lower()
    company_lower = company.lower() if company else ""

    title   = (result.get("title")   or "").lower()
    snippet = (result.get("snippet") or "").lower()
    link    = (result.get("link")    or "").lower()

    score = 0.0

    # ── Name matching ─────────────────────────────────
    name_parts = name_lower.split()
    matched_parts = sum(1 for p in name_parts if p in title)
    score += (matched_parts / max(len(name_parts), 1)) * 50  # up to 50 pts

    # Bonus: exact full name in title
    if name_lower in title:
        score += 15

    # ── Company matching ──────────────────────────────
    if company_lower:
        if company_lower in title or company_lower in snippet:
            score += 25
        # Partial company word match
        for word in company_lower.split():
            if len(word) > 3 and word in title + snippet:
                score += 5
                break

    # ── URL is a LinkedIn profile (not company/school page) ──
    if re.search(r"linkedin\.com/in/", link):
        score += 10

    return min(score, 100.0)


# ─────────────────────────────────────────────
#  Main class
# ─────────────────────────────────────────────

class LinkedInFinder:
    """
    Find a person's LinkedIn profile URL given their full name and company.

    Parameters
    ----------
    api_key : str, optional
        Your SerpApi key. Falls back to the ``SERPAPI_API_KEY`` environment
        variable.  Pass ``"DEMO"`` to use built-in mock data for testing.
    pause_between_requests : float
        Seconds to wait between API calls (avoids rate-limiting). Default 1.0.
    """

    BASE_URL   = "https://serpapi.com/search"
    ACCOUNT_URL = "https://serpapi.com/account"

    def __init__(
        self,
        api_key: Optional[str] = None,
        pause_between_requests: float = 1.0,
    ):
        self.api_key  = api_key or os.getenv("SERPAPI_API_KEY", "")
        self.pause    = pause_between_requests
        self._use_mock = (not self.api_key) or (self.api_key.upper() == "DEMO")

        if self._use_mock:
            print("⚠️  No SerpApi key found – running in DEMO/mock mode.\n"
                  "   Results are fabricated for testing purposes.\n"
                  "   Set SERPAPI_API_KEY to use live data.")

    # ──────────────────────────────────────────
    #  Public API
    # ──────────────────────────────────────────

    def find_profile(
        self,
        full_name:  str,
        company:    str = "",
        *,
        max_pages:  int  = 2,
        min_score:  float = 30.0,
        verbose:    bool  = True,
    ) -> dict:
        """
        Search for a LinkedIn profile URL.

        Parameters
        ----------
        full_name   : The person's full name, e.g. ``"Jane Smith"``.
        company     : Company name (strongly recommended for accuracy).
        max_pages   : How many Google result pages to scan (1–5).
        min_score   : Minimum confidence score (0–100) to consider a match.
        verbose     : Print progress messages.

        Returns
        -------
        dict with keys:
            ``found``        – bool
            ``url``          – LinkedIn URL or ``None``
            ``name``         – matched name from result title
            ``score``        – confidence 0–100
            ``snippet``      – result snippet text
            ``all_candidates`` – list of all scored candidates
        """
        if not full_name.strip():
            raise ValueError("full_name must not be empty.")

        max_pages = max(1, min(max_pages, 5))

        if verbose:
            print(f"\n🔍  Searching LinkedIn for: {full_name!r}"
                  + (f"  |  Company: {company!r}" if company else ""))

        candidates = []

        for page in range(1, max_pages + 1):
            if verbose:
                print(f"    Page {page}/{max_pages} …", end=" ", flush=True)

            raw = self._raw_search(full_name, company, page=page)

            if "error" in raw:
                if verbose:
                    print(f"❌  API error: {raw['error']}")
                break

            results = raw.get("organic_results", [])
            if not results:
                if verbose:
                    print("no results.")
                break

            for r in results:
                link = r.get("link", "")
                if not re.search(r"linkedin\.com/in/", link):
                    continue
                sc = _score_result(r, full_name, company)
                candidates.append({
                    "url":     link,
                    "name":    _clean(r.get("title", "").split(" - ")[0]),
                    "score":   round(sc, 1),
                    "snippet": _clean(r.get("snippet", "")),
                })

            if verbose:
                print(f"found {len(results)} results, "
                      f"{sum(1 for c in candidates if c['score'] >= min_score)} profile(s) above threshold.")

            time.sleep(self.pause)

        # Sort by score descending
        candidates.sort(key=lambda x: x["score"], reverse=True)

        best = candidates[0] if candidates and candidates[0]["score"] >= min_score else None

        if verbose:
            if best:
                print(f"\n✅  Best match  [{best['score']}/100]: {best['url']}")
                print(f"    Name    : {best['name']}")
                print(f"    Snippet : {best['snippet'][:120]}…" if len(best['snippet']) > 120 else f"    Snippet : {best['snippet']}")
            else:
                print("\n❌  No confident match found.")
                if candidates:
                    print(f"    Top candidate (score {candidates[0]['score']}/100, below threshold {min_score}): "
                          f"{candidates[0]['url']}")

        return {
            "found":          bool(best),
            "url":            best["url"]     if best else None,
            "name":           best["name"]    if best else None,
            "score":          best["score"]   if best else 0.0,
            "snippet":        best["snippet"] if best else "",
            "all_candidates": candidates,
        }

    def batch_find(
        self,
        people:    list[dict],
        *,
        max_pages: int   = 2,
        min_score: float = 30.0,
        verbose:   bool  = True,
    ) -> list[dict]:
        """
        Find LinkedIn URLs for multiple people at once.

        Parameters
        ----------
        people : list of dicts with keys ``name`` (required) and ``company``
                 (optional).  Example::

                     [
                         {"name": "Jane Smith",  "company": "Acme Corp"},
                         {"name": "John Doe",    "company": ""},
                     ]

        Returns
        -------
        List of result dicts (same structure as ``find_profile``), each
        augmented with the original ``input_name`` and ``input_company``.
        """
        results = []
        total   = len(people)

        for i, person in enumerate(people, 1):
            name    = person.get("name", "").strip()
            company = person.get("company", "").strip()

            if not name:
                if verbose:
                    print(f"[{i}/{total}] Skipping empty name.")
                results.append({"found": False, "url": None, "input_name": name,
                                 "input_company": company, "score": 0.0,
                                 "name": None, "snippet": "", "all_candidates": []})
                continue

            if verbose:
                print(f"\n[{i}/{total}] ─────────────────────────────────────")

            res = self.find_profile(name, company, max_pages=max_pages,
                                    min_score=min_score, verbose=verbose)
            res["input_name"]    = name
            res["input_company"] = company
            results.append(res)

        return results

    def account_info(self) -> dict:
        """Return SerpApi account / quota information."""
        if self._use_mock:
            return {"plan_name": "DEMO", "plan_searches_left": 999,
                    "account_email": "demo@example.com", "total_searches_used": 0}
        try:
            r = requests.get(self.ACCOUNT_URL,
                             params={"api_key": self.api_key}, timeout=10)
            return r.json() if r.ok else {"error": r.text}
        except Exception as e:
            return {"error": str(e)}

    # ──────────────────────────────────────────
    #  Internal helpers
    # ──────────────────────────────────────────

    def _build_query(self, full_name: str, company: str) -> str:
        """Craft a precise Google query for LinkedIn profile pages."""
        parts = [f'site:linkedin.com/in/', f'"{full_name}"']
        if company:
            parts.append(f'"{company}"')
        return " ".join(parts)

    def _raw_search(self, full_name: str, company: str, page: int = 1) -> dict:
        """Execute one search page; returns raw API dict (or mock)."""
        if self._use_mock:
            return self._mock_response(full_name, company)

        query = self._build_query(full_name, company)
        params = {
            "engine":  "google",
            "q":       query,
            "start":   (page - 1) * 10,
            "num":     10,
            "api_key": self.api_key,
        }

        try:
            r = requests.get(self.BASE_URL, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 401:
                return {"error": "Invalid API key (401 Unauthorized)."}
            elif r.status_code == 429:
                return {"error": "Rate limit hit (429). Try again later."}
            else:
                return {"error": f"HTTP {r.status_code}: {r.text[:200]}"}
        except requests.exceptions.Timeout:
            return {"error": "Request timed out."}
        except Exception as e:
            return {"error": str(e)}

    # ──────────────────────────────────────────
    #  Mock data (no API key needed for testing)
    # ──────────────────────────────────────────

    def _mock_response(self, full_name: str, company: str) -> dict:
        slug = full_name.lower().replace(" ", "-")
        co   = company.lower().replace(" ", "")
        return {
            "search_metadata":   {"status": "Success (mock)"},
            "search_parameters": {"q": self._build_query(full_name, company)},
            "organic_results": [
                {
                    "title":   f"{full_name} - Founder & CEO at {company or 'Tech Co'}",
                    "link":    f"https://www.linkedin.com/in/{slug}-{co[:6] or 'demo'}",
                    "snippet": (
                        f"{full_name} is the Founder and CEO at {company or 'an innovative startup'}. "
                        "Previously worked at Google and Microsoft. "
                        "Passionate about AI and building great teams."
                    ),
                },
                {
                    "title":   f"{full_name} – Product Manager | LinkedIn",
                    "link":    f"https://www.linkedin.com/in/{slug}-pm",
                    "snippet": (
                        f"View {full_name}'s profile on LinkedIn, the world's largest "
                        "professional community."
                    ),
                },
                {
                    "title":   f"Contact {full_name} – {company or 'Tech'} | LinkedIn",
                    "link":    f"https://www.linkedin.com/company/{co[:10] or 'techco'}",
                    "snippet": "Company page – not a personal profile.",
                },
            ],
        }


# ─────────────────────────────────────────────
#  Colab-friendly interactive runner
# ─────────────────────────────────────────────

def colab_search(api_key: str = ""):
    """
    Interactive prompt loop for Google Colab.
    Call this function in a cell to start a guided search session.
    """
    print("=" * 58)
    print("  🔗  LinkedIn Profile Finder  –  Powered by SerpApi")
    print("=" * 58)

    finder = LinkedInFinder(api_key=api_key or None)

    info = finder.account_info()
    if "plan_searches_left" in info:
        print(f"\n📊  Account  : {info.get('account_email', 'N/A')}")
        print(f"    Plan     : {info.get('plan_name', 'N/A')}")
        print(f"    Searches left: {info.get('plan_searches_left', 'N/A')}\n")

    results_log = []

    while True:
        print("\n" + "─" * 58)
        full_name = input("👤  Full name  (or 'quit' to exit): ").strip()
        if full_name.lower() in {"quit", "exit", "q", ""}:
            break

        company = input("🏢  Company    (press Enter to skip): ").strip()

        res = finder.find_profile(full_name, company)
        results_log.append(res)

        print("\n── Would you like to see all candidates? (y/n) ", end="")
        if input().strip().lower() == "y":
            print()
            for idx, c in enumerate(res["all_candidates"], 1):
                print(f"  {idx}. [{c['score']:>5.1f}/100] {c['url']}")
                print(f"       {c['name']}")
                print(f"       {c['snippet'][:90]}…" if len(c['snippet']) > 90 else f"       {c['snippet']}")

        another = input("\n🔄  Search another person? (y/n): ").strip().lower()
        if another != "y":
            break

    print("\n✨  Session complete.")
    return results_log


# ─────────────────────────────────────────────
#  Quick self-test (run this file directly)
# ─────────────────────────────────────────────

if __name__ == "__main__":
    finder = LinkedInFinder()                  # uses DEMO mode if no key set

    # Single lookup
    result = finder.find_profile("Satya Nadella", "Microsoft")
    print("\nResult dict:", json.dumps({k: v for k, v in result.items()
                                        if k != "all_candidates"}, indent=2))

    # Batch lookup
    batch = finder.batch_find([
        {"name": "Jensen Huang",   "company": "NVIDIA"},
        {"name": "Sam Altman",     "company": "OpenAI"},
        {"name": "",               "company": "skip me"},   # bad row – gracefully skipped
    ])
    print(f"\nBatch: {len(batch)} results, "
          f"{sum(1 for r in batch if r['found'])} found.")